# Mortality Modelling Using ONS Deaths and Population Data (2015–2021)

A complete data-cleaning and modelling pipeline combining ONS deaths and population estimates to produce age‑specific mortality rates for regions in the UK.

Here we fetch our data using the Nomis API (National Online Manpower Information System Appliation Processing Interface) from the ONS (Office for Nation Statistic website). This allows to get the data directly from the website.
Link: https://www.nomisweb.co.uk/api/v01/help 

In [146]:
# Fetch mortality data from ONS API

import pandas as pd

url = (
    "https://www.nomisweb.co.uk/api/v01/dataset/NM_161_1.data.csv?geography=2013265921...2013265930,2013265933&date=latestMINUS9,latestMINUS8,latestMINUS7,latestMINUS6,latestMINUS5,latestMINUS4,latestMINUS3&cause_of_death=0&gender=0...2&age=0...20&measure=1&measures=20100"
)
print(url)
mortality_df = pd.read_csv(url)



https://www.nomisweb.co.uk/api/v01/dataset/NM_161_1.data.csv?geography=2013265921...2013265930,2013265933&date=latestMINUS9,latestMINUS8,latestMINUS7,latestMINUS6,latestMINUS5,latestMINUS4,latestMINUS3&cause_of_death=0&gender=0...2&age=0...20&measure=1&measures=20100


In [147]:
mortality_df.head()

,DATE,DATE_NAME,DATE_CODE,DATE_TYPE,DATE_TYPECODE,DATE_SORTORDER,GEOGRAPHY,GEOGRAPHY_NAME,GEOGRAPHY_CODE,GEOGRAPHY_TYPE,...,MEASURES,MEASURES_NAME,OBS_VALUE,OBS_STATUS,OBS_STATUS_NAME,OBS_CONF,OBS_CONF_NAME,URN,RECORD_OFFSET,RECORD_COUNT
0,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,28075,A,Normal Value,F,Free (free for publication),Nm-161d1d32240e0d2013265921d98a99d0d0d1d20100,0,4410
1,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,95,A,Normal Value,F,Free (free for publication),Nm-161d1d32240e0d2013265921d98a99d0d1d1d20100,1,4410
2,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,18,A,Normal Value,F,Free (free for publication),Nm-161d1d32240e0d2013265921d98a99d0d2d1d20100,2,4410
3,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,5,A,Normal Value,F,Free (free for publication),Nm-161d1d32240e0d2013265921d98a99d0d3d1d20100,3,4410
4,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,21,A,Normal Value,F,Free (free for publication),Nm-161d1d32240e0d2013265921d98a99d0d4d1d20100,4,4410


In [148]:
# Extract the relevant colums

mortality_cols = [ 
    "DATE", # dates are from 2015 to 2021
    "AGE_NAME", # we will rename this to "age" later for clarity, it is the age band of the deceased.
    "GENDER",
    "GEOGRAPHY_CODE", # we use the geography code as it is easier to join datasets using this.
    "GEOGRAPHY_NAME",
    "OBS_VALUE", #  this is the number of deaths, we will rename it to "deaths" later for clarity.
]

clean_mortality_df = mortality_df[mortality_cols].copy()

# Rename columns for clarity

#use a dictionary to rename the columns, this is more readable and less error prone than renaming each column individually.
clean_mortality_df = clean_mortality_df.rename(columns={
    "DATE": "year",
    "AGE_NAME": "age",
    "GENDER": "gender", # ONS uses 0 for all, 1 for male and 2 for female
    "GEOGRAPHY_CODE": "geo",
    "GEOGRAPHY_NAME": "geo_name",
    "OBS_VALUE": "deaths",
})[["year", "age", "gender", "geo", "geo_name", "deaths"]] # gives a new dataframe with only the columns we want, in the order we want.


clean_mortality_df = clean_mortality_df[clean_mortality_df["age"] != "total (all ages)"]
clean_mortality_df.head()



,year,age,gender,geo,geo_name,deaths
1,2015,Aged under 1,0,E12000001,North East,95
2,2015,Aged 1 to 4,0,E12000001,North East,18
3,2015,Aged 5 to 9,0,E12000001,North East,5
4,2015,Aged 10-14,0,E12000001,North East,21
5,2015,Aged 15-19,0,E12000001,North East,40


In [149]:
clean_mortality_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4200 entries, 1 to 4409
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   year      4200 non-null   int64 
 1   age       4200 non-null   object
 2   gender    4200 non-null   int64 
 3   geo       4200 non-null   object
 4   geo_name  4200 non-null   object
 5   deaths    4200 non-null   int64 
dtypes: int64(3), object(3)
memory usage: 229.7+ KB


In [150]:
# Fetch population data from ONS API

url = (
    "https://www.nomisweb.co.uk/api/v01/dataset/NM_2002_1.data.csv?geography=2013265921...2013265932&date=latestMINUS9,latestMINUS8,latestMINUS7,latestMINUS6,latestMINUS5,latestMINUS4,latestMINUS3,latest&gender=0...2&c_age=200,101...191&measures=20100"
)
print(url)
population_df = pd.read_csv(url)

population_df.head()

https://www.nomisweb.co.uk/api/v01/dataset/NM_2002_1.data.csv?geography=2013265921...2013265932&date=latestMINUS9,latestMINUS8,latestMINUS7,latestMINUS6,latestMINUS5,latestMINUS4,latestMINUS3,latest&gender=0...2&c_age=200,101...191&measures=20100


,DATE,DATE_NAME,DATE_CODE,DATE_TYPE,DATE_TYPECODE,DATE_SORTORDER,GEOGRAPHY,GEOGRAPHY_NAME,GEOGRAPHY_CODE,GEOGRAPHY_TYPE,...,MEASURES,MEASURES_NAME,OBS_VALUE,OBS_STATUS,OBS_STATUS_NAME,OBS_CONF,OBS_CONF_NAME,URN,RECORD_OFFSET,RECORD_COUNT
0,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,2611804,A,Normal Value,F,Free (free for publication),Nm-2002d1d32240e2d2013265921d0d200d20100,0,26496
1,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,28278,A,Normal Value,F,Free (free for publication),Nm-2002d1d32240e2d2013265921d0d101d20100,1,26496
2,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,29042,A,Normal Value,F,Free (free for publication),Nm-2002d1d32240e2d2013265921d0d102d20100,2,26496
3,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,29886,A,Normal Value,F,Free (free for publication),Nm-2002d1d32240e2d2013265921d0d103d20100,3,26496
4,2015,2015,2015,date,0,0,2013265921,North East,E12000001,regions,...,20100,Value,30796,A,Normal Value,F,Free (free for publication),Nm-2002d1d32240e2d2013265921d0d104d20100,4,26496


In [151]:
population_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   DATE                 25000 non-null  int64 
 1   DATE_NAME            25000 non-null  int64 
 2   DATE_CODE            25000 non-null  int64 
 3   DATE_TYPE            25000 non-null  object
 4   DATE_TYPECODE        25000 non-null  int64 
 5   DATE_SORTORDER       25000 non-null  int64 
 6   GEOGRAPHY            25000 non-null  int64 
 7   GEOGRAPHY_NAME       25000 non-null  object
 8   GEOGRAPHY_CODE       25000 non-null  object
 9   GEOGRAPHY_TYPE       25000 non-null  object
 10  GEOGRAPHY_TYPECODE   25000 non-null  int64 
 11  GEOGRAPHY_SORTORDER  25000 non-null  int64 
 12  GENDER               25000 non-null  int64 
 13  GENDER_NAME          25000 non-null  object
 14  GENDER_CODE          25000 non-null  int64 
 15  GENDER_TYPE          25000 non-null  object
 16  GEND

In [152]:
# Extract the relevant colums
population_df = population_df[population_df["C_AGE"] != 200]
population_df["C_AGE_CODE"]= population_df["C_AGE"] - 101
population_cols = [ 
    "DATE", # dates are from 2015 to 2021
    "C_AGE_CODE", # we will rename this to "age" later for clarity, it is the age band of the population.
    "GENDER",
    "GEOGRAPHY_CODE", # we use the geography code as it is easier to join datasets using this.
    "GEOGRAPHY_NAME",
    "OBS_VALUE", #  this is the number of deaths, we will rename it to "deaths" later for clarity.
]

clean_population_df = population_df[population_cols].copy()

# Rename columns for clarity

#use a dictionary to rename the columns, this is more readable and less error prone than renaming each column individually.
clean_population_df = clean_population_df.rename(columns={
    "DATE": "year",
    "C_AGE_CODE": "age",
    "GENDER": "gender", # ONS uses 0 for all, 1 for male and 2 for female
    "GEOGRAPHY_CODE": "geo",
    "GEOGRAPHY_NAME": "geo_name",
    "OBS_VALUE": "population",
})[["year", "age", "gender", "geo", "geo_name", "population"]] # gives a new dataframe with only the columns we want, in the order we want.

clean_population_df.head()


,year,age,gender,geo,geo_name,population
1,2015,0,0,E12000001,North East,28278
2,2015,1,0,E12000001,North East,29042
3,2015,2,0,E12000001,North East,29886
4,2015,3,0,E12000001,North East,30796
5,2015,4,0,E12000001,North East,31081


In [153]:
bins = [0, 1, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50,
        55, 60, 65, 70, 75, 80, 85, 90, 200]

labels = [
    "Aged under 1", "Aged 1 to 4", "Aged 5 to 9", "Aged 10-14", "Aged 15-19",
    "Aged 20-24", "Aged 25-29", "Aged 30-34", "Aged 35-39", "Aged 40-44",
    "Aged 45-49", "Aged 50-54", "Aged 55-59", "Aged 60-64", "Aged 65-69",
    "Aged 70-74", "Aged 75-79", "Aged 80-84", "Aged 85-89", "Aged 90 and over"
]

clean_population_df["age"] = pd.cut(
    clean_population_df["age"],
    bins=bins,  
    labels=labels,
    right=False
)


In [154]:
clean_population_df.head()

,year,age,gender,geo,geo_name,population
1,2015,Aged under 1,0,E12000001,North East,28278
2,2015,Aged 1 to 4,0,E12000001,North East,29042
3,2015,Aged 1 to 4,0,E12000001,North East,29886
4,2015,Aged 1 to 4,0,E12000001,North East,30796
5,2015,Aged 1 to 4,0,E12000001,North East,31081


In [155]:
clean_population_df.head()

,year,age,gender,geo,geo_name,population
1,2015,Aged under 1,0,E12000001,North East,28278
2,2015,Aged 1 to 4,0,E12000001,North East,29042
3,2015,Aged 1 to 4,0,E12000001,North East,29886
4,2015,Aged 1 to 4,0,E12000001,North East,30796
5,2015,Aged 1 to 4,0,E12000001,North East,31081


In [156]:
clean_population_banded = clean_population_df.groupby(
    ["year", "geo", "geo_name", "gender", "age"],
    observed=True,
    as_index=False
)["population"].sum()


In [157]:
clean_mortality_df["age"] = clean_mortality_df["age"].replace({
    "Aged 0-4": "Aged under 1",   # if needed
    "Aged 1-4": "Aged 1 to 4",
    "Aged 5-9": "Aged 5 to 9",
    "Aged 10-14": "Aged 10-14",
    "Aged 15-19": "Aged 15-19",
    "Aged 20-24": "Aged 20-24",
    "Aged 25-29": "Aged 25-29",
    "Aged 30-34": "Aged 30-34",
    "Aged 35-39": "Aged 35-39",
    "Aged 40-44": "Aged 40-44",
    "Aged 45-49": "Aged 45-49",
    "Aged 50-54": "Aged 50-54",
    "Aged 55-59": "Aged 55-59",
    "Aged 60-64": "Aged 60-64",
    "Aged 65-69": "Aged 65-69",
    "Aged 70-74": "Aged 70-74",
    "Aged 75-79": "Aged 75-79",
    "Aged 80-84": "Aged 80-84",
    "Aged 85+": "Aged 85-89",     # if needed
    "Aged 90+": "Aged 90 and over"
})


In [158]:
merged_df = clean_mortality_df.merge(
    clean_population_banded,
    on=["year", "geo", "geo_name", "gender", "age"],
    how="inner"
)


In [159]:
merged_df.head()

,year,age,gender,geo,geo_name,deaths,population
0,2015,Aged under 1,0,E12000001,North East,95,28278
1,2015,Aged 1 to 4,0,E12000001,North East,18,120805
2,2015,Aged 5 to 9,0,E12000001,North East,5,149769
3,2015,Aged 10-14,0,E12000001,North East,21,135822
4,2015,Aged 15-19,0,E12000001,North East,40,154486


In [160]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4200 entries, 0 to 4199
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   year        4200 non-null   int64 
 1   age         4200 non-null   object
 2   gender      4200 non-null   int64 
 3   geo         4200 non-null   object
 4   geo_name    4200 non-null   object
 5   deaths      4200 non-null   int64 
 6   population  4200 non-null   int64 
dtypes: int64(4), object(3)
memory usage: 229.8+ KB


# extract the relevant columns

population_col = [ 
    "DATE", # dates are from 2015 to 2021
    "C_AGE_CODE", # this is the age code, we will convert it to age later for clarity.
    "GENDER",
    "GEOGRAPHY_CODE", # we use the geography code as it is easier to join datasets using this.
    "GEOGRAPHY_NAME",
    "OBS_VALUE", # this is the population, we will rename it to "population" later for clarity.
]


clean_population_df = population_df[population_col].copy()

# Rename columns for clarity

#use a dictionary to rename the columns, this is more readable and less error prone than renaming each column individually.
clean_population_df = clean_population_df.rename(columns={
    "DATE": "year",
    "GENDER": "gender", # ONS uses 0 for all, 1 for male and 2 for female
    "C_AGE_CODE": "age",
    "GEOGRAPHY_CODE": "geo",
    "GEOGRAPHY_NAME": "geo_name",
    "OBS_VALUE": "population",
})
clean_population_df = clean_population_df[clean_population_df["age"] != 200] # remove age code 200 which is "all ages", we don't need this as we have the age specific population data.
clean_population_df["age"] = clean_population_df["age"] - 101 # convert age code to age, ONS uses 101 for age 1 
clean_population_df = clean_population_df[["year", "gender","age", "geo", "geo_name", "population"]] # gives a new dataframe with only the columns we want, in the order we want.


clean_population_df.head()


We merge the mortality and population dataframes on year, gender, age and geography code, we will use pandas default inner join as we want to keep all the mortality data even if there is no population data for some rows.

merged_df = clean_mortality_df.merge(clean_population_df, on=["year", "gender", "age","geo",])
merged_df = merged_df.drop(columns=[ "geo_name_y"]) # drop the geo_name columns as they are the same in both dataframes, we only need one of them.
merged_df = merged_df.rename(columns={"geo_name_x": "geo_name"}) # rename the geo_name column to geo_name for clarity.

merged_df.head(10)


Here we can see no missing values so the data merged successfully.

In [161]:
merged_df.to_csv("../data/ons_mortality_merged.csv", index=False)


## Saving the Cleaned and Merged Dataset

The data preparation stage is now complete.  
We have:

- cleaned the ONS deaths dataset  
- cleaned the ONS population dataset  
- removed non‑age categories (e.g., “All Ages”)  
- aligned age, gender, year, and geography  
- merged both datasets into a single, analysis‑ready table  
- verified that all rows match with no missing values  

The final dataset has been saved as `ons_mortality_merged.csv` and will be used in the next notebook for actuarial modelling.